# 00 — Define Region Polygon

Define the study area and coffee-farm labels for the Kona Coffee Belt.

Coffee farm polygons come from the **Hawaii 2020 Agricultural Land Use Baseline**
(Hawaii Statewide GIS Program).  592 polygons labeled `Crops_2020 = 'Coffee'` fall
within the Kona bounding box — no manual drawing needed.

Outputs:
- `../data/polygons/kona_region.geojson` — bounding polygon for the study area
- `../data/polygons/kona_coffee_farms.geojson` — 592 real coffee-farm polygons (label = 1)

In [1]:
import geopandas as gpd
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import os

OUT_DIR  = '../data/polygons'
SHP_PATH = '../data/polygons/aglanduse_2020.shp'

os.makedirs(OUT_DIR, exist_ok=True)

/home/simonhans/anaconda3/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


## Region boundary

Derived from the actual coffee farm polygons — buffered union of all Kona + Ka'ū parcels.
Using union (not convex hull) preserves the natural U-shape around Mauna Loa's west and
south slopes without filling in the empty mountain interior.

In [2]:
# Load coffee farms first (needed to derive the region boundary)
ag = gpd.read_file(SHP_PATH).to_crs('EPSG:4326')

# Kona (west slope) + Ka'ū (south slope) — both on Mauna Loa's flanks
# Lat ceiling 19.80 excludes the 3 stray Waimea/Hamakua polygons at 20.03–20.09°N
MAUNA_LOA_BBOX = (-156.10, 18.90, -155.40, 19.80)  # minx, miny, maxx, maxy

coffee_all = ag[ag['Crops_2020'] == 'Coffee']
farms_gdf  = coffee_all.cx[
    MAUNA_LOA_BBOX[0]:MAUNA_LOA_BBOX[2],
    MAUNA_LOA_BBOX[1]:MAUNA_LOA_BBOX[3]
].copy()
farms_gdf  = farms_gdf.reset_index(drop=True)
farms_gdf['label'] = 1

print(f'Coffee polygons (Kona + Kaʻū): {len(farms_gdf)}')
print(f'Total acreage:                 {farms_gdf["Acreage"].sum():.1f} acres')
print(f'Farm bounds: lon {farms_gdf.total_bounds[0]:.4f} to {farms_gdf.total_bounds[2]:.4f}')
print(f'             lat {farms_gdf.total_bounds[1]:.4f} to {farms_gdf.total_bounds[3]:.4f}')

# Build region: buffered union of farms (no convex hull) — preserves U-shape
farms_utm  = farms_gdf.to_crs('EPSG:32604')
region_utm = farms_utm.unary_union.buffer(2000)  # 2 km buffer
region_gdf = gpd.GeoDataFrame(
    {'name': ['mauna_loa_belt'], 'geometry': [region_utm]}, crs='EPSG:32604'
).to_crs('EPSG:4326')

region_gdf.to_file(os.path.join(OUT_DIR, 'kona_region.geojson'), driver='GeoJSON')
farms_gdf.to_file(os.path.join(OUT_DIR, 'kona_coffee_farms.geojson'), driver='GeoJSON')
print('Saved kona_region.geojson')
print('Saved kona_coffee_farms.geojson')

Coffee polygons (Kona + Kaʻū): 674
Total acreage:                 5986.0 acres
Farm bounds: lon -155.9859 to -155.4568
             lat 19.0606 to 19.7399
Saved kona_region.geojson
Saved kona_coffee_farms.geojson


In [5]:
import geopandas as gpd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from pyproj import Transformer

DEM_PATH   = '../data/DEM/hawaii_DEM_500m_utm.tif'   # full island, EPSG:32604
FARMS_PATH = '../data/polygons/kona_coffee_farms.geojson'
IMG = '../img'
BUF = 0.06   # degrees (~6 km) padding around farms

# ── Load farms (native EPSG:4326) ────────────────────────────────────────────
farms = gpd.read_file(FARMS_PATH)
b = farms.total_bounds              # (lon_min, lat_min, lon_max, lat_max)

# ── Load full-island DEM (UTM) for contour computation ───────────────────────
with rasterio.open(DEM_PATH) as src:
    dem = src.read(1).astype(float)
    if src.nodata is not None:
        dem[dem == src.nodata] = np.nan
    transform = src.transform
    h, w = dem.shape

# Build UTM coordinate grid
utm_x = np.array([transform.c + (j + 0.5) * transform.a for j in range(w)])
utm_y = np.array([transform.f + (i + 0.5) * transform.e for i in range(h)])
UTM_X, UTM_Y = np.meshgrid(utm_x, utm_y)

# Convert UTM → lat/lon
t = Transformer.from_crs('EPSG:32604', 'EPSG:4326', always_xy=True)
LON, LAT = t.transform(UTM_X.ravel(), UTM_Y.ravel())
LON = LON.reshape(UTM_X.shape)
LAT = LAT.reshape(UTM_X.shape)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 11))

# Minor contours every 100 m
ax.contour(LON, LAT, dem, levels=np.arange(-50, 4300, 100),
           colors='#bbbbbb', linewidths=0.4, zorder=1)
# Major contours every 500 m, labeled
cs_major = ax.contour(LON, LAT, dem, levels=np.arange(0, 4300, 500),
                      colors='#666666', linewidths=0.9, zorder=2)
ax.clabel(cs_major, fmt='%d m', fontsize=11, inline=True)

# ── Farms ─────────────────────────────────────────────────────────────────────
farms.plot(ax=ax, facecolor='#ff7700', edgecolor='white',
           linewidth=0.15, alpha=0.9, zorder=5)

# ── Mauna Loa summit ──────────────────────────────────────────────────────────
ax.plot(-155.5908, 19.4754, marker='^', color='black', markersize=10,
        markeredgecolor='#555555', markeredgewidth=0.5, zorder=7)
ax.annotate('Mauna Loa\n4169 m', (-155.5908, 19.4754), xytext=(10, 4),
            textcoords='offset points', fontsize=12,
            fontweight='bold', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'),
            zorder=8)

# ── District labels ───────────────────────────────────────────────────────────
for (x, y, label) in [(-155.91, 19.57, 'Kona'), (-155.57, 19.15, 'Ka\u02bbu')]:
    ax.text(x, y, label, color='#cc5500', fontsize=17, fontweight='bold',
            ha='center', va='center', zorder=7,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      alpha=0.7, edgecolor='#ff7700', linewidth=0.8))

# ── Legend ────────────────────────────────────────────────────────────────────
farm_patch = mpatches.Patch(facecolor='#ff7700', edgecolor='white',
                             linewidth=0.5, label=f'Coffee farms  (n={len(farms)})')
ax.legend(handles=[farm_patch], loc='lower right', fontsize=12,
          edgecolor='#aaaaaa', framealpha=0.9)

# ── North arrow ───────────────────────────────────────────────────────────────
ax.annotate('', xy=(0.96, 0.97), xytext=(0.96, 0.91),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', color='black', lw=1.8))
ax.text(0.96, 0.975, 'N', transform=ax.transAxes,
        ha='center', va='bottom', fontsize=13, fontweight='bold')

# ── Crop to study area + buffer ───────────────────────────────────────────────
ax.set_xlim(b[0] - BUF, b[2] + BUF)
ax.set_ylim(b[1] - BUF, b[3] + BUF)

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.set_xlabel('Longitude', fontsize=12, labelpad=6)
ax.set_ylabel('Latitude', fontsize=12, labelpad=6)
ax.tick_params(labelsize=11)

ax.set_title('Hawai\u02bbi Island \u2014 Kona and Ka\u02bbu Coffee Districts',
             fontsize=15, fontweight='bold', pad=14)

plt.tight_layout()
plt.savefig(f'{IMG}/00_region.png', dpi=300, bbox_inches='tight')
print('Saved 00_region.png')
plt.show()

/home/simonhans/anaconda3/lib/python3.7/site-packages/geopandas/plotting.py:51: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if geom is not None and geom.type.startswith(prefix) and not geom.is_empty:


Saved 00_region.png


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:97: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


In [ ]:
# Export region as KML for upload to USGS TNM downloader
# https://apps.nationalmap.gov/downloader/ → Upload File → search for 1m DEM tiles
from shapely.geometry import MultiPolygon, Polygon

geom = region_gdf.geometry.iloc[0]
polys = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]

placemark_blocks = []
for i, poly in enumerate(polys):
    coords = ' '.join(f'{x},{y},0' for x, y in poly.exterior.coords)
    placemark_blocks.append(f'''  <Placemark>
    <name>mauna_loa_belt_{i}</name>
    <Polygon>
      <outerBoundaryIs>
        <LinearRing>
          <coordinates>{coords}</coordinates>
        </LinearRing>
      </outerBoundaryIs>
    </Polygon>
  </Placemark>''')

kml = '<?xml version="1.0" encoding="UTF-8"?>\n'
kml += '<kml xmlns="http://www.opengis.net/kml/2.2">\n<Document>\n'
kml += '\n'.join(placemark_blocks)
kml += '\n</Document>\n</kml>'

kml_path = os.path.join(OUT_DIR, 'kona_region.kml')
with open(kml_path, 'w') as f:
    f.write(kml)

print(f'Saved {kml_path}  ({len(polys)} polygon parts)')